# The Uneven Curb: Explainer Notebook

**DTU Social Visualization (course 02806). Final project. FY2025.**

This notebook is the long version of our project. The website is the short version. The website link is https://syedtaimurhassan.github.io/02806-project/site/

The notebook has 8 sections:

1. Motivation
2. Basic Stats
3. Data Analysis
4. Genre
5. Visualizations
6. Discussion
7. Contributions
8. References

The website is for a normal reader. It is short. The notebook is for the teachers. It shows the work behind the page.

The data file is `Parking_Violations_Issued_-_Fiscal_Year_2025.csv` from NYC Open Data. The file is around 3 GB. We do not put it in the repo. The link is in Section 8.


## 1. Motivation

One parking ticket is small. But sixteen million tickets in one year is a big record of how the city handles the curb.

We picked NYC parking violations for three reasons.

**1. The data is big and free.** It is on NYC Open Data. The file has more than 16 million rows. Each row is one ticket. There are many columns: date, time, violation code, precinct, borough, plate type, registration state, issuing agency. So we can ask many questions on the same file.

**2. The data has a hidden problem.** There is a code in the file called `Precinct 0`. Almost half the rows have this code. It is not a real precinct. It is the label for camera tickets. So a normal map mixes camera tickets and officer tickets. We will explain this in Section 3.

**3. Parking tickets are everyday life.** People in NYC know them. So a story about them is easy for any reader. The website does not need a science background.

### Our question

> If we remove the camera tickets, how uneven is the rest of the system across precincts, rule types, time of day, and money value?

### What we do not ask

We do not ask if the system is fair or unfair. The data does not have what we need for that. We do not have:

- tickets that should have been given but were not
- tickets that were paid in full
- tickets that were dropped or reduced
- the number of cars parked legally vs illegally
- income, race, or background of the drivers

So we only show the shape of the issued tickets. We do not say what caused it.

### Goal for the reader

The website is for a friend who has not taken this class. After 5 minutes the friend should learn three things:

1. The data has two systems inside. Camera tickets and officer tickets are very different.
2. The officer side is not equal. Some places get many more tickets. The rule mix also changes by neighborhood.
3. Counting tickets is not the same as counting money. The hour of the day also matters.

The notebook is the long version of the same story for the teachers.


## 2. Basic Stats

This section sets up the project and shows the shape of the file.

The raw CSV is around 3 GB. It is too big to load all at once on a normal laptop. So we read it in chunks of 500 thousand rows. We do not keep the chunks. We only count things and write small summary files into `outputs/eda/`. After this step we never read the raw file again. We only read the small summaries.

This makes the rest of the work fast. Every chart in this notebook is built from the small CSVs and from JSON files in `outputs/story/`. Most cells run in less than a second.


### Imports and constants

The first cell sets up the libraries. We use `pandas` for tables and `plotly.express` for charts. We set the plotly renderer to `iframe_connected` so figures show in the classic Jupyter Notebook server (and also in VS Code, JupyterLab, etc.). With this renderer, plotly writes each figure as a small HTML file in `iframe_figures/` and embeds it. This avoids the JavaScript loading issue that often hides figures in classic Jupyter.

The path to the raw CSV is here. The file is not in the repo because it is too big.

We also pick the fiscal year dates. NYC fiscal year 2025 is from 1 July 2024 to 30 June 2025. We will drop any row outside this range.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import urllib.request

import pandas as pd
import plotly.express as px
import plotly.io as pio
from plotly.offline import init_notebook_mode

# Make figures show in classic Jupyter Notebook server in the browser.
# iframe_connected writes each figure to ./iframe_figures/ and embeds it.
# This is the most compatible option: works in classic Jupyter, JupyterLab,
# VS Code, and nbviewer without any frontend setup.
pio.renderers.default = "iframe_connected"
init_notebook_mode(connected=True)

DATA_DIR = Path('.')
CSV_PATH = DATA_DIR / 'Parking_Violations_Issued_-_Fiscal_Year_2025.csv'
OUTPUT_DIR = DATA_DIR / 'outputs'
EDA_DIR = OUTPUT_DIR / 'eda'
STORY_DIR = OUTPUT_DIR / 'story'
OUTPUT_DIR.mkdir(exist_ok=True)
EDA_DIR.mkdir(exist_ok=True)
STORY_DIR.mkdir(exist_ok=True)

FY_START = pd.Timestamp('2024-07-01')
FY_END = pd.Timestamp('2025-06-30')
CHUNKSIZE = 500_000


def load_json(path):
    with open(path) as f:
        return json.load(f)

if CSV_PATH.exists():
    print(True, f'{CSV_PATH.stat().st_size / 1_000_000:.1f} MB')
else:
    print(False, 'Raw CSV not found locally; existing outputs/eda summaries can still be used, but rebuilding them requires the NYC Open Data CSV.')


The second cell has the constants we use across the notebook.

The borough map is needed because the file uses many short codes for the same borough. For example `K`, `BK`, and `KINGS` all mean Brooklyn. The map gives one clean borough name for each code.

`CAMERA_CODES` is a small set of codes we knew were camera-only at the start. Later we used a better rule based on `Precinct 0`. So this set is just a first guess.

`BOROUGHS` is the list of valid borough names. We use it to filter out junk values like `Newy` or `Unknown`.


In [ ]:
BOROUGH_MAP = {
    'NY': 'Manhattan', 'MN': 'Manhattan', 'MAN': 'Manhattan', 'NEW YORK': 'Manhattan',
    'K': 'Brooklyn', 'BK': 'Brooklyn', 'KINGS': 'Brooklyn',
    'Q': 'Queens', 'QN': 'Queens', 'QNS': 'Queens', 'QUEEN': 'Queens', 'QUEENS': 'Queens',
    'BX': 'Bronx', 'BRONX': 'Bronx',
    'R': 'Staten Island', 'ST': 'Staten Island', 'RICH': 'Staten Island', 'STATEN ISLAND': 'Staten Island', 'RICHMOND': 'Staten Island',
}
CAMERA_CODES = {'7', '12', '36'}
BOROUGHS = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
WEEKDAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


def normalize_borough(x):
    if pd.isna(x):
        return 'Unknown'
    key = str(x).strip().upper()
    return BOROUGH_MAP.get(key, key.title() if key else 'Unknown')


def parse_hour(s):
    s = s.astype(str).str.strip().str.upper()
    h = pd.to_numeric(s.str[:2], errors='coerce')
    m = pd.to_numeric(s.str[2:4], errors='coerce')
    ap = s.str[4:5]
    valid = h.between(1, 12) & m.between(0, 59) & ap.isin(['A', 'P'])
    h24 = h.mask((ap == 'A') & (h == 12), 0).mask((ap == 'P') & (h < 12), h + 12)
    return h24.where(valid)

### Cleaning and chunked reading

This is the function that reads the raw CSV in chunks and writes the small CSVs. We do not run it again here because we already ran it once. The output files are in `outputs/eda/`. We just show the function so the cleaning logic is visible.

The function does these steps in order:

1. Read the CSV in chunks of 500 thousand rows.
2. Parse `Issue Date` as a date.
3. Drop rows with bad dates.
4. Drop rows outside the fiscal year window.
5. Standardize the borough name.
6. For each chunk, count things into `Counter` objects: codes, hours, weekdays, precincts, states, and so on.
7. Save each `Counter` as a small CSV in `outputs/eda/`.

We removed about 1,470 rows with bad dates. We removed about 307,482 rows outside FY2025. The final number of rows is 16,250,291.


In [ ]:
def build_eda_tables():
    out = OUTPUT_DIR / 'eda'
    out.mkdir(exist_ok=True)
    usecols = ['Issue Date', 'Violation Time', 'Violation Code', 'Violation Description',
               'Violation County', 'Violation Precinct', 'Registration State', 'Plate Type',
               'Issuing Agency', 'Vehicle Body Type', 'Vehicle Make']

    quality = Counter()
    borough = Counter(); borough_camera = Counter(); borough_curb = Counter()
    codes = Counter(); months = Counter(); hours = Counter(); weekdays = Counter(); hour_weekday = Counter()
    precincts = Counter(); states = Counter(); plates = Counter(); agencies = Counter(); makes = Counter()

    for chunk in pd.read_csv(CSV_PATH, usecols=usecols, dtype=str, chunksize=CHUNKSIZE, low_memory=False):
        quality['raw_rows'] += len(chunk)
        dt = pd.to_datetime(chunk['Issue Date'], errors='coerce', format='%m/%d/%Y')
        quality['invalid_dates'] += int(dt.isna().sum())
        in_fy = dt.between(FY_START, FY_END)
        quality['outside_fy'] += int((~in_fy & dt.notna()).sum())
        chunk = chunk.loc[in_fy].copy()
        dt = dt.loc[in_fy]

        chunk['borough'] = chunk['Violation County'].map(normalize_borough)
        chunk['code'] = chunk['Violation Code'].astype(str).str.strip()
        chunk['camera'] = chunk['code'].isin(CAMERA_CODES)
        chunk['hour'] = parse_hour(chunk['Violation Time'])
        chunk['weekday'] = dt.dt.day_name()
        chunk['month'] = dt.dt.to_period('M').astype(str)
        chunk['precinct'] = chunk['Violation Precinct'].astype(str).str.strip()

        quality['fy_rows'] += len(chunk)
        quality['precinct_zero'] += int(chunk['precinct'].eq('0').sum())
        quality['missing_borough'] += int(chunk['borough'].eq('Unknown').sum())
        quality['missing_hour'] += int(chunk['hour'].isna().sum())

        borough.update(chunk['borough']); borough_camera.update(chunk.loc[chunk['camera'], 'borough']); borough_curb.update(chunk.loc[~chunk['camera'], 'borough'])
        codes.update(zip(chunk['code'], chunk['Violation Description'].fillna('Unknown')))
        months.update(chunk['month']); hours.update(chunk['hour'].dropna().astype(int)); weekdays.update(chunk['weekday']); hour_weekday.update(zip(chunk['weekday'], chunk['hour'].dropna().astype(int)))
        precincts.update(chunk['precinct']); states.update(chunk['Registration State'].fillna('Unknown')); plates.update(chunk['Plate Type'].fillna('Unknown'))
        agencies.update(chunk['Issuing Agency'].fillna('Unknown')); makes.update(chunk['Vehicle Make'].fillna('Unknown'))

    borough_df = pd.DataFrame({'borough': sorted(set(borough) | set(borough_camera) | set(borough_curb))})
    borough_df['tickets'] = borough_df['borough'].map(borough).fillna(0).astype(int)
    borough_df['camera_tickets'] = borough_df['borough'].map(borough_camera).fillna(0).astype(int)
    borough_df['curbside_tickets'] = borough_df['borough'].map(borough_curb).fillna(0).astype(int)
    borough_df['camera_share'] = borough_df['camera_tickets'] / borough_df['tickets']

    pd.DataFrame([quality]).to_csv(out / 'quality.csv', index=False)
    borough_df.to_csv(out / 'borough.csv', index=False)
    pd.DataFrame([(a,b,c) for (a,b),c in codes.items()], columns=['code','description','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'codes.csv', index=False)
    pd.DataFrame(months.items(), columns=['month','tickets']).sort_values('month').to_csv(out / 'months.csv', index=False)
    pd.DataFrame(hours.items(), columns=['hour','tickets']).sort_values('hour').to_csv(out / 'hours.csv', index=False)
    pd.DataFrame(weekdays.items(), columns=['weekday','tickets']).to_csv(out / 'weekdays.csv', index=False)
    pd.DataFrame([(d,h,c) for (d,h),c in hour_weekday.items()], columns=['weekday','hour','tickets']).to_csv(out / 'hour_weekday.csv', index=False)
    pd.DataFrame(precincts.items(), columns=['precinct','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'precincts.csv', index=False)
    pd.DataFrame(states.items(), columns=['state','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'states.csv', index=False)
    pd.DataFrame(plates.items(), columns=['plate_type','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'plate_types.csv', index=False)
    pd.DataFrame(agencies.items(), columns=['agency','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'agencies.csv', index=False)
    pd.DataFrame(makes.items(), columns=['make','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'makes.csv', index=False)

# Build summaries only if they do not already exist.
# The full FY2025 CSV is too large for GitHub, so a clean clone should either
# download the raw CSV first or restore outputs/eda before running this notebook.
if not (EDA_DIR / 'quality.csv').exists():
    if not CSV_PATH.exists():
        raise FileNotFoundError(
            f'Missing {EDA_DIR / "quality.csv"} and missing raw CSV {CSV_PATH}. '
            'Download the FY2025 NYC Parking Violations CSV or restore outputs/eda before running this notebook.'
        )
    build_eda_tables()


### Load the small summary tables

After the cleaning pass we load the small CSVs. Every chart from now on uses these tables, not the raw file.


In [ ]:
EDA_DIR = OUTPUT_DIR / 'eda'
quality = pd.read_csv(EDA_DIR / 'quality.csv')
borough = pd.read_csv(EDA_DIR / 'borough.csv')
codes = pd.read_csv(EDA_DIR / 'codes.csv')
months = pd.read_csv(EDA_DIR / 'months.csv')
hours = pd.read_csv(EDA_DIR / 'hours.csv')
weekdays = pd.read_csv(EDA_DIR / 'weekdays.csv')
hour_weekday = pd.read_csv(EDA_DIR / 'hour_weekday.csv')
precincts = pd.read_csv(EDA_DIR / 'precincts.csv')
states = pd.read_csv(EDA_DIR / 'states.csv')
plates = pd.read_csv(EDA_DIR / 'plate_types.csv')
agencies = pd.read_csv(EDA_DIR / 'agencies.csv')
makes = pd.read_csv(EDA_DIR / 'makes.csv')

quality

### Top-level numbers

The next cell shows the basic shape of the file. It uses `quality.csv` from the cleaning step.

Important numbers:

- **Raw rows:** 16,559,243
- **Rows kept:** 16,250,291
- **Bad date rows dropped:** 1,470
- **Rows outside FY2025 dropped:** 307,482
- **Precinct 0 rows:** 7,524,683 (about 46.3 percent)

`precinct_zero_share` is one of the most important numbers in the project. About 46 percent of all rows have `Violation Precinct = 0`. This is not a real precinct. Section 3 explains what it really is.


In [ ]:
# Scale: the draft's "billion ticket machine" framing starts with total volume and camera share.
scale = quality.assign(
    precinct_zero_share = quality['precinct_zero'] / quality['fy_rows'],
    invalid_date_share = quality['invalid_dates'] / quality['raw_rows']
)
scale.T

### Summons number integrity check

A summons number is the unique ID of a ticket. If a row has a duplicate or missing summons number, it is not a clean record.

We checked this. All 16,250,291 rows have a unique summons number. Zero duplicates. So row count = ticket count.

This means we can say "16.25 million tickets" without an asterisk.


In [ ]:
summons_quality = pd.read_csv(EDA_DIR / 'summons_quality.csv')
summons_dist = pd.read_csv(EDA_DIR / 'summons_count_distribution.csv')
summons_metrics = load_json(STORY_DIR / 'summons_metrics.json')

summons_quality


## 3. Data Analysis

This section walks through the analysis in the order we did it. Each step taught us something. The next step was decided by what the last step showed.

The path is:

1. First look. What is in the data.
2. The Precinct 0 problem. Why the first map was wrong.
3. Real precincts only. The map after we removed Precinct 0.
4. Fine value. Tickets are not all worth the same money.
5. Timing by channel. Cameras and officers do not work at the same hours.
6. Violation families. Group the 90 codes into a smaller list.
7. Precinct specialization. Different precincts have different top rules.
8. Family fine value. Add the dollar layer at family level.
9. Repeat vehicles. Some cars get many tickets.
10. Daily anomalies. The last six days of FY2025 look strange.

We show the chart for each step and write what it tells us.


### 3.1 First look

We start simple. What are the most common violations? Where do the tickets go? When do they happen?

The next table is the top 12 violation descriptions. The column `tickets` is the count for the whole year.

The biggest code is school zone speed. This is a camera code. Next are bus lane and red light. Also cameras. Then street cleaning, which is an officer code. So we already see two systems mixing in the top rows.


In [ ]:
codes.head(12)

In [ ]:
fig = px.bar(codes.head(12).sort_values('tickets'), x='tickets', y='description', orientation='h',
             title='Top violation descriptions in FY2025')
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

This bar chart splits each borough total into camera tickets and officer tickets. We used a simple guess for camera here. Codes 7, 12, and 36 were treated as camera. Later we used a better rule based on `Precinct 0`.

Brooklyn and Queens have the most tickets. Manhattan is third. The split is different in each borough. Manhattan is mostly officer. Staten Island is mostly camera. This is the first hint that the geographic story is two stories.


In [ ]:
plot_borough = borough[borough['borough'].isin(BOROUGHS)].sort_values('tickets', ascending=False)
fig = px.bar(plot_borough, x='borough', y=['curbside_tickets', 'camera_tickets'],
             title='Tickets by borough: curbside vs camera-like', labels={'value':'Tickets', 'variable':'Type'})
fig

The monthly line shows volume across the year. The shape is mostly flat. There is a small dip in winter. There is a sharp drop at the very end of June 2025. We explain this in step 10. It is a data cutoff, not a real drop.


In [ ]:
fig = px.line(months, x='month', y='tickets', markers=True, title='Monthly ticket volume across FY2025')
fig.update_layout(xaxis_title='', yaxis_title='Tickets')
fig

The hour-by-weekday heatmap shows when tickets are written. The dark band is mid-morning on weekdays. This matches alternate side parking, when streets are cleaned and officers walk the curb. Weekend volume is lower because most alternate side rules pause on Sundays.

This view also looks like one system. We will see in step 5 that it is two systems with different peaks. The mid-morning peak hides this.


In [ ]:
heat = hour_weekday.pivot(index='weekday', columns='hour', values='tickets').reindex(WEEKDAYS)
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Viridis',
                title='Ticket timing: weekday by hour')
fig.update_layout(xaxis_title='Hour of day', yaxis_title='')
fig

The top registration states are New York, New Jersey, Pennsylvania, and a few more. New York plates are the biggest by far. This is normal. Most cars in NYC have NY plates.


In [ ]:
fig = px.bar(states.head(10).sort_values('tickets'), x='tickets', y='state', orientation='h',
             title='Top registration states')
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

### 3.2 The Precinct 0 problem

When we tried to map tickets by precinct number, the map looked strange. One precinct had a huge number. That precinct was 0.

There is no NYPD precinct 0. The number is not on any map. So what is it?

Below we list the top codes with `Violation Precinct = 0`. The first cell loads two small tables: one for codes inside Precinct 0, one for codes inside the rest. The second cell shows the top 12 codes for Precinct 0.


In [ ]:
code_by_borough = pd.read_csv(EDA_DIR / 'code_by_borough.csv', dtype={'code': str})
precinct_zero_codes = pd.read_csv(EDA_DIR / 'precinct_zero_codes.csv', dtype={'code': str})
nonzero_precinct_codes = pd.read_csv(EDA_DIR / 'nonzero_precinct_codes.csv', dtype={'code': str})
channel_by_borough = pd.read_csv(EDA_DIR / 'channel_by_borough.csv')

code_by_borough.head()

In [ ]:
# What is inside precinct 0?
precinct_zero_codes.head(12)

The top codes inside Precinct 0 are school zone speed, bus lane, red light, MTA bus camera double parking, and more. These are all camera codes. The pattern is very clean.

So Precinct 0 is the label the system uses when there is no real precinct. This happens for camera tickets because there is no officer in the field. The camera takes the picture and the system fills in `0`.

The chart below shows this for the top ten codes.


In [ ]:
fig = px.bar(
    precinct_zero_codes.head(10).sort_values('tickets'),
    x='tickets', y='description', orientation='h',
    title='Top violations recorded with precinct 0'
)
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

For comparison, here are the top codes in the rest of the precincts (where the precinct number is real and not zero).


In [ ]:
fig = px.bar(
    nonzero_precinct_codes.head(10).sort_values('tickets'),
    x='tickets', y='description', orientation='h',
    title='Top violations recorded with non-zero precincts'
)
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

The two lists are almost completely different. The Precinct 0 list is camera-only. The real precinct list is street cleaning, no parking, hydrant, double parking, sticker, meter, and so on. These are all officer codes.

So the dataset is really two datasets stacked together. We must separate them before any map makes sense.

The next chart shows how much of each borough is camera (Precinct 0) versus officer (real precincts).


In [ ]:
# Borough dependence on camera-like enforcement.
channel_plot = channel_by_borough[channel_by_borough['borough'].isin(BOROUGHS)].sort_values('camera_share')
fig = px.bar(channel_plot, x='camera_share', y='borough', orientation='h',
             title='Share of borough tickets from camera-like enforcement')
fig.update_layout(xaxis_tickformat='.0%', xaxis_title='Camera-like share', yaxis_title='')
fig

This bar shows that camera enforcement is not equal across boroughs. Staten Island and Queens are heavily camera. Manhattan is much more officer. The borough story we showed before mixed these two together. After this point, every map drops Precinct 0 from the geography part.


### 3.3 Real precincts only

Once Precinct 0 is removed, the precinct map works. It is not perfect. Some precinct numbers in the data are not in the GeoJSON file. Those are unmappable. The mappable real precinct count is 78.

The cells below load the precinct table and the GeoJSON file from NYC Open Data, then keep only the precincts that match the GeoJSON.


In [ ]:
precincts = pd.read_csv(EDA_DIR / 'precincts.csv', dtype={'precinct': str})
fine_rank = pd.read_csv(EDA_DIR / 'fine_rank_comparison.csv', dtype={'code': str})

precincts.head(), fine_rank.head()

In [ ]:
PRECINCT_GEOJSON_URL = 'https://data.cityofnewyork.us/resource/y76i-bdw7.geojson?$limit=5000'
PRECINCT_GEOJSON_PATH = OUTPUT_DIR / 'police_precincts.geojson'

if not PRECINCT_GEOJSON_PATH.exists():
    urllib.request.urlretrieve(PRECINCT_GEOJSON_URL, PRECINCT_GEOJSON_PATH)

with open(PRECINCT_GEOJSON_PATH) as f:
    precinct_geojson = json.load(f)

geo_precincts = {feature['properties']['precinct'] for feature in precinct_geojson['features']}
real_precincts = precincts[precincts['precinct'].ne('0')].copy()
map_precincts = real_precincts[real_precincts['precinct'].isin(geo_precincts)].copy()
missing_precincts = sorted(set(real_precincts['precinct']) - geo_precincts)

print('Mappable precincts:', len(map_precincts))
print('Non-mappable non-zero precinct labels:', len(missing_precincts))
print(missing_precincts[:20])

In [ ]:
fig = px.choropleth_map(
    map_precincts,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='properties.precinct',
    color='tickets',
    hover_name='precinct',
    hover_data={'tickets': ':,'},
    color_continuous_scale='Viridis',
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.82,
    title='FY2025 parking violations by real NYPD precinct, excluding precinct 0'
)
fig.update_traces(marker_line_width=0.7, marker_line_color='white')
fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.write_html(OUTPUT_DIR / 'eda_precinct_nonzero_map.html')
fig

This map is the first valid geographic view. The Manhattan precincts are the darkest. Precinct 19 (Upper East Side) is on top. Precinct 14 (Midtown South) is next. Then Precinct 13. Then Precinct 6. Then Precinct 1. Five of the top six are Manhattan.

This is not a surprise. Manhattan has the most metered curb, the most deliveries, and the most foot traffic from officers. A higher number does not mean drivers there are worse. It means enforcement is concentrated there.


### 3.4 Estimated fine value

A ticket count treats every ticket the same. But not every ticket has the same dollar amount. The dataset does not include the fine column. So we estimated the fine using the violation code, the rule book from NYC Department of Finance, and the data dictionary.

The cell below loads the fine table and shows the top codes by estimated fine value.

Important wording: this is **estimated fine value from issued tickets**. It is not revenue. The city does not collect every dollar on every ticket. Some are paid, some reduced, some dropped, some never paid. The dataset does not have the payment outcome.


In [ ]:
# Estimated fines by violation code.
# This uses the data dictionary's fine table. It is approximate because exact fine zone is not in our summary.
fines = pd.read_csv(EDA_DIR / 'fine_estimates.csv', dtype={'code': str})
fines[['code', 'violation_description_dict', 'tickets', 'fine_other_areas', 'estimated_fines_other_areas']].head(12)

In [ ]:
fig = px.bar(
    fines.head(10).sort_values('estimated_fines_other_areas'),
    x='estimated_fines_other_areas', y='violation_description_dict', orientation='h',
    title='Top violation codes by estimated fine value',
    labels={'estimated_fines_other_areas': 'Estimated fines ($)', 'violation_description_dict': ''}
)
fig

The scatter below puts ticket count on one axis and estimated fine value on the other. Each point is one violation code.

Most codes line up because more tickets = more dollars. But some are off the line. School zone speed is huge by both. Some hydrant codes are smaller in count but bigger in dollars per ticket.


In [ ]:
fig = px.scatter(
    fine_rank.head(20),
    x='tickets', y='estimated_fines_other_areas', text='code',
    hover_name='violation_description_dict',
    title='Ticket volume vs estimated fine value, top 20 fine-value codes',
    labels={'tickets': 'Tickets', 'estimated_fines_other_areas': 'Estimated fine value ($)'}
)
fig.update_traces(textposition='top center')
fig

In [ ]:
# Categories whose money rank is much higher than their ticket-count rank.
fine_rank.sort_values('rank_shift', ascending=False).head(10)[[
    'code', 'violation_description_dict', 'tickets', 'fine_other_areas',
    'estimated_fines_other_areas', 'ticket_rank', 'value_rank', 'rank_shift'
]]

The table above shows the codes that move up the most when we switch from count rank to dollar rank. These codes punish more per ticket. Hydrant and bus lane stand out.

The dollar layer changes which precincts look biggest. We will see this again at the precinct level in Section 5.


### 3.5 Timing by enforcement channel

The hour-by-weekday heatmap in step 1 looked like one system. Once we split Precinct 0 from real precincts, the timing also splits.

The cell below loads the channel-by-hour table and the channel-by-weekday table.


In [ ]:
channel_hour = pd.read_csv(EDA_DIR / 'channel_hour.csv')
channel_weekday = pd.read_csv(EDA_DIR / 'channel_weekday.csv')
channel_month = pd.read_csv(EDA_DIR / 'channel_month.csv')
channel_hour_weekday = pd.read_csv(EDA_DIR / 'channel_hour_weekday.csv')

channel_hour.head()

The line chart below shows the share of each channel's daily tickets by hour. The two channels have very different shapes.

Real precinct tickets peak at 9 in the morning. This is when alternate side parking is enforced. Officers walk the curb in front of the street sweepers.

Precinct 0 tickets peak in the afternoon, around 3 PM. This is when bus-lane cameras and bus-stop cameras work the hardest. Afternoon rush, more buses, more drivers blocking bus stops.

So the same dataset has two different daily clocks inside. One for officers, one for cameras.


In [ ]:
channel_hour['channel_total'] = channel_hour.groupby('channel')['tickets'].transform('sum')
channel_hour['share'] = channel_hour['tickets'] / channel_hour['channel_total']

fig = px.line(channel_hour, x='hour', y='share', color='channel', markers=True,
              title='Hourly ticket profile by enforcement channel',
              labels={'hour': 'Hour of day', 'share': 'Share of channel tickets', 'channel': ''})
fig.update_layout(yaxis_tickformat='.1%')
fig.write_html(STORY_DIR / 'fig4_channel_hour_profile.html')
fig

The weekday chart is similar. Real precinct tickets peak on weekdays (Tuesday is the highest). Precinct 0 tickets are spread across the week, including Sundays. Cameras run all the time. Officers follow alternate side schedules.


In [ ]:
channel_weekday['weekday'] = pd.Categorical(channel_weekday['weekday'], categories=WEEKDAYS, ordered=True)
channel_weekday = channel_weekday.sort_values(['channel', 'weekday'])
channel_weekday['channel_total'] = channel_weekday.groupby('channel')['tickets'].transform('sum')
channel_weekday['share'] = channel_weekday['tickets'] / channel_weekday['channel_total']

fig = px.bar(channel_weekday, x='weekday', y='share', color='channel', barmode='group',
             title='Weekday ticket profile by enforcement channel',
             labels={'weekday': '', 'share': 'Share of channel tickets', 'channel': ''})
fig.update_layout(yaxis_tickformat='.1%')
fig.write_html(STORY_DIR / 'fig5_channel_weekday_profile.html')
fig

### 3.6 Violation families

NYC has more than 90 violation codes. This is too many for a public reader. So we group them into a smaller list called violation families. There are about ten families.

Examples:

- Street cleaning
- Meter / paid parking
- No standing / no parking
- Hydrant
- Double parking
- Registration / inspection sticker
- Bus lane / bus stop
- School-zone speed
- Red light
- MTA camera double parking
- Other

The grouping is in `family_totals.csv` and related tables in `outputs/eda/`. The cell below loads the family tables and shows the volume by family.


In [ ]:
family_totals = pd.read_csv(EDA_DIR / 'family_totals.csv')
family_channel_summary = pd.read_csv(EDA_DIR / 'family_channel_summary.csv')
family_borough = pd.read_csv(EDA_DIR / 'family_borough.csv')
family_hour = pd.read_csv(EDA_DIR / 'family_hour.csv')

family_totals

In [ ]:
fig = px.bar(family_totals.sort_values('tickets'), x='tickets', y='family', orientation='h',
             title='Violation families by ticket volume')
fig.update_layout(xaxis_title='Tickets', yaxis_title='')
fig.write_html(STORY_DIR / 'fig8_violation_family_totals.html')
fig

The biggest family is school zone speed. Then bus lane. Then street cleaning. Then no parking. Then meter. The top two are camera-only. Below them, the rest are officer-driven.

The next chart shows what share of each family is in Precinct 0. This asks: how much of this family is camera based?


In [ ]:
fig = px.bar(family_channel_summary.sort_values('precinct0_share'), x='precinct0_share', y='family',
             orientation='h', color='precinct0_share', color_continuous_scale='Viridis',
             title='How much of each violation family is precinct 0?')
fig.update_layout(xaxis_tickformat='.0%', xaxis_title='Precinct 0 share', yaxis_title='', coloraxis_showscale=False)
fig.write_html(STORY_DIR / 'fig9_family_precinct0_share.html')
fig

The pattern is clear. School-zone speed, bus lane, red light, and MTA camera double parking are almost completely in Precinct 0. They are camera families.

Street cleaning, no parking, meter, hydrant, sticker, and double parking are almost completely in real precincts. They are officer families.

So the camera-vs-officer split is not just about Precinct 0. It is also visible at the family level. The two sets almost do not overlap.


The next chart shows family share by borough. We use the top 8 families.


In [ ]:
top_families = family_totals.head(8)['family'].tolist()
family_borough_plot = family_borough[
    family_borough['borough'].isin(BOROUGHS) & family_borough['family'].isin(top_families)
]

fig = px.bar(family_borough_plot, x='borough', y='tickets', color='family', barmode='stack',
             title='Top violation families by borough')
fig.update_layout(xaxis_title='', yaxis_title='Tickets', legend_title='')
fig.write_html(STORY_DIR / 'fig10_family_by_borough.html')
fig

Manhattan is mostly meters and no standing. Brooklyn and Bronx are heavier on street cleaning. Staten Island stands out for sticker tickets. We explain why in step 7.


The final chart in this step is hourly profiles for the top families. Each line shows the share of that family's daily tickets at each hour.


In [ ]:
family_hour_plot = family_hour[family_hour['family'].isin(top_families)].copy()
family_hour_plot['family_total'] = family_hour_plot.groupby('family')['tickets'].transform('sum')
family_hour_plot['share'] = family_hour_plot['tickets'] / family_hour_plot['family_total']

fig = px.line(family_hour_plot, x='hour', y='share', color='family',
              title='Hourly profiles of major violation families')
fig.update_layout(yaxis_tickformat='.1%', xaxis_title='Hour of day', yaxis_title='Share within family', legend_title='')
fig.write_html(STORY_DIR / 'fig11_family_hour_profiles.html')
fig

The lines have very different shapes. Street cleaning peaks at 9 AM. Hydrant peaks at 6 AM. Sticker peaks at 8 AM. Meter and no parking and double parking peak around 1 PM. Bus lane peaks at 4 PM. School zone peaks at noon.

There is no single "parking ticket hour". Each family has its own schedule.


### 3.7 Precinct specialization

Volume is one thing. Mix is another. Two precincts can have the same total tickets but be very different inside.

We computed a specialization ratio for each (precinct, family) pair:

```
ratio = (family share inside this precinct) / (family share across all real precincts)
```

A ratio of 1 means the family is normal here. A ratio of 2 means the family is twice as common here than across the city. A ratio above 5 means the precinct is very specialized for that rule.


In [ ]:
family_precinct = pd.read_csv(EDA_DIR / 'family_precinct.csv', dtype={'precinct': str})
dominant_family = pd.read_csv(EDA_DIR / 'dominant_family_by_precinct.csv', dtype={'precinct': str})
family_specialization = pd.read_csv(EDA_DIR / 'family_precinct_specialization.csv', dtype={'precinct': str})

family_precinct.head()

In [ ]:
fig = px.choropleth_map(
    dominant_family,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='properties.precinct',
    color='family',
    hover_name='precinct',
    hover_data={'tickets': ':,', 'family_share_in_precinct': ':.1%', 'precinct_total': ':,'},
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.85,
    title='Dominant violation family by mapped NYPD precinct'
)
fig.update_traces(marker_line_width=0.7, marker_line_color='white')
fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0}, legend_title='Dominant family')
fig.write_html(STORY_DIR / 'fig12_dominant_family_by_precinct.html')
fig

The map above colors each precinct by its top family. We can see the pattern by neighborhood.

- Most of the city is street cleaning (44 of 78 precincts).
- A band in Manhattan is meters (12 precincts).
- Some precincts are no standing (12 precincts).
- A small group is registration sticker (5 precincts).
- A few are mixed (5 precincts grouped under Other).

The next chart focuses on one family, street cleaning, to show how its share changes across precincts.


In [ ]:
street_cleaning = family_precinct[family_precinct['family'].eq('Street cleaning')]
fig = px.choropleth_map(
    street_cleaning,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='properties.precinct',
    color='family_share_in_precinct',
    hover_name='precinct',
    hover_data={'tickets': ':,', 'family_share_in_precinct': ':.1%', 'specialization_ratio': ':.2f'},
    color_continuous_scale='Viridis',
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.85,
    title='Street-cleaning share within each mapped precinct'
)
fig.update_traces(marker_line_width=0.7, marker_line_color='white')
fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0}, coloraxis_colorbar={'tickformat': '.0%'})
fig.write_html(STORY_DIR / 'fig13_street_cleaning_precinct_share.html')
fig

The next chart shows the strongest specialization examples by ratio.


In [ ]:
fig = px.bar(
    family_specialization.head(15).sort_values('specialization_ratio'),
    x='specialization_ratio', y='precinct', color='family', orientation='h',
    title='Strongest precinct specializations by violation family',
    hover_data={'tickets': ':,', 'family_share_in_precinct': ':.1%'}
)
fig.update_layout(xaxis_title='Specialization ratio vs mapped-precinct average', yaxis_title='Precinct', legend_title='Family')
fig.write_html(STORY_DIR / 'fig14_precinct_family_specialization.html')
fig

The strongest ratio is Precinct 123 with sticker tickets. Ratio 5.54. Precinct 123 is the South Shore of Staten Island. Far from transit, very car-dependent. So sticker tickets dominate.

Precinct 34 (Washington Heights and Inwood) has a high double parking ratio. Around 3.0. The blocks are narrow, deliveries are common, and there is not enough legal curb. So drivers double park.

Precinct 44 (South Bronx near Yankee Stadium) is similar.

These are not the precincts with the most tickets. They are the precincts with the most unusual mix.


### 3.8 Family fine value

The fine value layer also works at family level. The cell below loads the family fine estimates.


In [ ]:
family_fine = pd.read_csv(EDA_DIR / 'family_fine_estimates.csv')
family_fine[[
    'family', 'tickets', 'estimated_fines_other_areas',
    'avg_estimated_fine_other_areas', 'ticket_rank', 'value_rank', 'rank_shift'
]]

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

plot = family_fine.sort_values('estimated_fines_other_areas')
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=('Tickets', 'Estimated fine value'))
fig.add_trace(go.Bar(x=plot['tickets'], y=plot['family'], orientation='h', marker_color='#247BA0'), row=1, col=1)
fig.add_trace(go.Bar(x=plot['estimated_fines_other_areas'], y=plot['family'], orientation='h', marker_color='#F25F5C'), row=1, col=2)
fig.update_layout(title='Violation families by ticket volume and estimated fine value', height=600, showlegend=False)
fig.write_html(STORY_DIR / 'fig15_family_count_vs_fine_value.html')
fig

The two bars are side by side. Left is ticket count. Right is estimated fine value.

School-zone speed is the biggest by both. But the ratio is not the same across families. Some families have a higher fine per ticket. The next chart shows the average fine per ticket by family.


In [ ]:
avg_fine = family_fine.dropna(subset=['avg_estimated_fine_other_areas']).sort_values('avg_estimated_fine_other_areas')
fig = px.bar(
    avg_fine,
    x='avg_estimated_fine_other_areas', y='family', orientation='h',
    title='Average estimated fine per ticket by violation family',
    hover_data={'tickets': ':,', 'known_fine_ticket_share': ':.1%'}
)
fig.update_layout(xaxis_title='Average estimated fine ($)', yaxis_title='')
fig.write_html(STORY_DIR / 'fig16_family_average_fine.html')
fig

Double parking has the highest average fine of the officer families. Around 115 dollars. So a precinct with heavy double parking enforcement looks bigger in dollars than in count.

This matters at precinct level. We will see in Section 5 that some precincts move up by 9 ranks when we switch from count to dollars. Others move down by 10.


### 3.9 Repeat vehicles

The dataset has a vehicle key. We did not store the actual plate. We only stored an anonymous hash so two tickets for the same plate can be matched without exposing the plate. This is for privacy.

We then asked: how many cars get many tickets, and how few cars carry most of the volume?


In [ ]:
repeat_distribution = pd.read_csv(EDA_DIR / 'repeat_vehicle_distribution.csv')
repeat_state = pd.read_csv(EDA_DIR / 'repeat_vehicle_by_state_group.csv')
repeat_plate = pd.read_csv(EDA_DIR / 'repeat_vehicle_by_plate_class.csv')
top_repeat_anonymous = pd.read_csv(EDA_DIR / 'top_repeat_vehicle_anonymous.csv')

repeat_distribution

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

bucket_order = ['1', '2', '3-5', '6-10', '11-25', '26-50', '51+']
repeat_distribution['ticket_bucket'] = pd.Categorical(repeat_distribution['ticket_bucket'], categories=bucket_order, ordered=True)
repeat_distribution = repeat_distribution.sort_values('ticket_bucket')

fig = make_subplots(rows=1, cols=2, subplot_titles=('Share of vehicle keys', 'Share of tickets'))
fig.add_trace(go.Bar(x=repeat_distribution['ticket_bucket'], y=repeat_distribution['vehicle_share'], marker_color='#2A9D8F'), row=1, col=1)
fig.add_trace(go.Bar(x=repeat_distribution['ticket_bucket'], y=repeat_distribution['ticket_share'], marker_color='#E76F51'), row=1, col=2)
fig.update_layout(title='Repeat vehicle concentration by ticket bucket', showlegend=False)
fig.update_yaxes(tickformat='.0%')
fig.write_html(STORY_DIR / 'fig25_repeat_vehicle_distribution.html')
fig

The two side-by-side bars are the heart of this section.

Left side. About 43 percent of cars have only one ticket. About 57 percent have more than one. Some have 26 or more. A small group has more than 100 in one year.

Right side. Cars with many tickets carry most of the volume. The 11 to 25 bucket alone has more than three million tickets. The 26 plus bucket has many more.

So the system is concentrated by car. About 56.6 percent of cars produce 89.2 percent of all tickets. The top 1 percent of cars carry 14.9 percent of the tickets. One anonymous car got 1,088 tickets in twelve months. About three a day, every day, for a year.

This is fleet behavior. Delivery trucks, livery cars, rideshare. The next chart shows tickets per vehicle key by plate class.


In [ ]:
repeat_plate_plot = repeat_plate[repeat_plate['vehicles'].ge(1000)].sort_values('tickets_per_vehicle')
fig = px.bar(repeat_plate_plot, x='tickets_per_vehicle', y='plate_class', orientation='h', color='repeat_vehicle_share',
             color_continuous_scale='Viridis', title='Tickets per vehicle key by plate class')
fig.update_layout(xaxis_title='Tickets per vehicle key', yaxis_title='', coloraxis_colorbar={'title': 'Repeat share', 'tickformat': '.0%'})
fig.write_html(STORY_DIR / 'fig27_tickets_per_vehicle_plate_class.html')
fig

Commercial and carrier plates are at the top. Passenger plates are much lower. So the very high counts come from commercial fleets, not normal drivers.

This finding was buried in the original notebook. We promoted it to its own section on the website.


### 3.10 Daily anomalies

The monthly chart in step 1 had a sharp drop at the end of June 2025. That drop is suspicious. The cell below loads the daily totals and looks for anomalies day by day.


In [ ]:
daily_total = pd.read_csv(EDA_DIR / 'daily_total.csv', parse_dates=['issue_date'])
daily_channel = pd.read_csv(EDA_DIR / 'daily_channel.csv', parse_dates=['issue_date'])
daily_anomalies = pd.read_csv(EDA_DIR / 'daily_total_anomalies_excluding_cutoff.csv', parse_dates=['issue_date'])
family_coverage = pd.read_csv(EDA_DIR / 'daily_family_coverage.csv')
daily_metrics = load_json(STORY_DIR / 'daily_anomaly_metrics.json')

daily_total.tail(10)


In [ ]:
channel_daily = daily_channel.pivot_table(index='issue_date', columns='channel', values='tickets', aggfunc='sum').sort_index().fillna(0)
channel_rolling = channel_daily.rolling(7, min_periods=1).mean().reset_index().melt(
    id_vars='issue_date', var_name='channel', value_name='tickets_7d_avg'
)

fig = px.line(channel_rolling, x='issue_date', y='tickets_7d_avg', color='channel',
              title='Daily ticket volume, 7-day rolling average')
fig.update_layout(xaxis_title='', yaxis_title='Tickets, 7-day average')
fig.write_html(STORY_DIR / 'fig51_daily_channel_rolling_average.html')
fig


The 7-day rolling line shows that real-precinct volume is steady through the year. Camera volume is also steady. But the last six days of FY2025 are much lower than the rolling average. This is a cutoff effect. The dataset was probably exported a few days after 30 June 2025, but late submissions had not arrived yet. So the very end of the file is incomplete.

We do not throw these days away. We just keep this in mind. We use full-year totals everywhere except this anomaly check.


## 4. Genre

This section maps our project onto Figure 7 of the Segel and Heer paper. The paper splits design into two halves. We answer each half.

### Primary genre choice

> **Magazine Style + Annotated Chart**, with **Martini Glass** narrative structure.

The website is one long page. The reader scrolls top to bottom. The page has a strong narrative voice. Each major figure is an Annotated Chart with a title, caption, and takeaway. The reader can hover and toggle inside the figure but cannot change the order of the page.

The Martini Glass shape means the start is tightly author-driven. We tell the reader where to look first. We give the headline numbers in the hero. We reveal Precinct 0 as a single moment. We map the real precincts. Near the bottom we open up. The user can hover, toggle legend items, and choose which family to look at in the hourly chart. So the stem of the glass is the prose. The wide mouth is the figure-level interactivity.

We picked this genre because the reader needs guidance through several traps. If they see a precinct map first, they will read it as if Precinct 0 is a real place. We have to walk the reader past that trap. A pure dashboard would let the reader make this mistake. A magazine-style page does not.

### Why this fit is right for our dataset

The dataset is layered. Camera vs officer. Volume vs mix. Volume vs dollars. Time of day vs rule type. A pure dashboard would dump all the layers at once. A pure linear video would not let the reader inspect detail.

A Magazine Style page with bounded interactivity is the right middle. We tell the reader the order. They can still hover and check our numbers. The Martini Glass shape lets us be strict at the top, where misreading risks are highest, and looser at the bottom, where the reader is on the same page as us.


## 5. Visualizations

The website tells the story with nine figures. This section walks through each one in the order it appears on the page. For every figure we say the chart type, why we chose it, and the main design choices. The website embeds the same HTML files via iframes; the cells below render them inline so the teachers can see them without leaving the notebook.

The full caption and one-line takeaway for every figure is in `outputs/story/final_figure_captions.md`.


### Figure 1. What gets ticketed (citywide totals)

This is the first figure on the website after the dataset section. It shows the size of each violation family for the whole year.

We picked a horizontal bar chart with one neutral color. The bars are sorted from biggest to smallest. We did not split by channel here. The point is just "what is the shape of the file" before we ask sharper questions. A horizontal layout makes the long family names readable without rotating labels.

We use the family grouping from Section 3.6, not the raw codes. Otherwise the chart would be dominated by hundreds of near-duplicate codes.


In [ ]:
family_totals = pd.read_csv(EDA_DIR / 'family_totals.csv')

fig = px.bar(
    family_totals.sort_values('tickets'),
    x='tickets', y='family', orientation='h',
    title='Citywide violation families by ticket count',
    labels={'tickets': 'Tickets', 'family': ''},
)
fig.update_layout(height=520)
fig.write_html(STORY_DIR / 'fig8_violation_family_totals.html')
fig


### Figure 2. Precinct 0 vs the real precincts

This figure is the first reveal of the project. It shows that Precinct 0 has a completely different rule mix than the real precincts.

We picked a stacked bar with two columns. One column is Precinct 0. The other is "Real precincts". Each bar stacks the family share inside that channel. The reader can see at a glance that the dominant families are not the same in the two columns.

We did not use side-by-side bars because we wanted each channel to feel like 100 percent of itself. The eye then compares the colored slices.

We sorted the family colors so the reader can move their eye top to bottom in the same order in both columns.


In [ ]:
precinct0_metrics = load_json(STORY_DIR / 'precinct0_narrative_metrics.json')
precinct0_family = pd.read_csv(EDA_DIR / 'precinct0_vs_real_precincts_family.csv')

precinct0_summary = pd.DataFrame([
    {
        'group': 'Precinct 0',
        'tickets': precinct0_metrics['precinct0_rows'],
        'share_of_fy2025': precinct0_metrics['precinct0_share'],
        'top_family': precinct0_metrics['top_precinct0_family'],
        'top_family_share': precinct0_metrics['top_precinct0_family_share'],
    },
    {
        'group': 'Real precincts',
        'tickets': precinct0_metrics['real_precinct_rows'],
        'share_of_fy2025': precinct0_metrics['real_precinct_share'],
        'top_family': precinct0_metrics['top_real_family'],
        'top_family_share': precinct0_metrics['top_real_family_share'],
    },
])
precinct0_summary


In [ ]:
family_order = (
    precinct0_family.groupby('family', as_index=False)['tickets'].sum()
    .sort_values('tickets', ascending=False)['family']
    .tolist()
)

fig = px.bar(
    precinct0_family,
    x='channel_label',
    y='share_within_channel',
    color='family',
    category_orders={'channel_label': ['Precinct 0', 'Real precincts'], 'family': family_order},
    custom_data=['tickets', 'share_within_channel', 'channel_total'],
    title='Precinct 0 is a different enforcement subsystem than real precincts',
    labels={'channel_label': '', 'share_within_channel': 'Share within channel', 'family': 'Violation family'},
)
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>%{legendgroup}<br>Tickets: %{customdata[0]:,}<br>Share in channel: %{customdata[1]:.1%}<br>Channel total: %{customdata[2]:,}<extra></extra>'
)
fig.update_layout(yaxis_tickformat='.0%', legend=dict(orientation='h', y=-0.32), height=620)
fig.write_html(STORY_DIR / 'fig_precinct0_vs_real_precincts.html')
fig


### Figure 3. Concentration curve

After Precinct 0 is removed, the next question is how uneven the rest of the system is. The concentration curve shows how much of the volume goes to how few of the precincts.

We used a Lorenz-style cumulative curve. The x axis is the cumulative share of precincts (sorted small to large). The y axis is the cumulative share of tickets. A diagonal would be perfect equality. Our curve bends well below the diagonal.

The numbers from the curve:

- Top 10 percent of precincts: 27.2 percent of tickets.
- Top 25 percent of precincts: 49.9 percent of tickets.

We added a dashed gray "equal distribution" line so the gap is easy to see. The hover tooltip shows the precinct number, rank, raw count, and the cumulative shares.

We picked this chart over a simple bar chart because the reader does not need to know each precinct number. They need to feel the unevenness.


In [ ]:
real_precinct_concentration = pd.read_csv(EDA_DIR / 'real_precinct_concentration_metrics.csv')
real_precinct_summary = load_json(STORY_DIR / 'real_precinct_concentration_summary.json')
real_precinct_summary


In [ ]:
import plotly.graph_objects as go

curve = real_precinct_concentration.copy()
curve['cumulative_precinct_share'] = curve['rank'] / len(curve)
curve = pd.concat([
    pd.DataFrame([{
        'precinct': 'start', 'tickets': 0, 'rank': 0,
        'cumulative_precinct_share': 0.0, 'cumulative_ticket_share': 0.0
    }]),
    curve
], ignore_index=True)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=curve['cumulative_precinct_share'],
    y=curve['cumulative_ticket_share'],
    mode='lines+markers',
    name='Real precinct ticket concentration',
    customdata=curve[['precinct', 'rank', 'tickets', 'cumulative_precinct_share', 'cumulative_ticket_share']],
    hovertemplate=(
        'Precinct: %{customdata[0]}<br>'
        'Rank: %{customdata[1]:.0f}<br>'
        'Tickets: %{customdata[2]:,}<br>'
        'Cumulative precinct share: %{customdata[3]:.1%}<br>'
        'Cumulative ticket share: %{customdata[4]:.1%}<extra></extra>'
    ),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines', name='Equal distribution',
    line=dict(color='gray', dash='dash')
))
fig.update_layout(
    title='A small share of precincts carries a large share of curbside tickets',
    xaxis_title='Cumulative share of real precincts',
    yaxis_title='Cumulative share of real-precinct tickets',
    xaxis_tickformat='.0%',
    yaxis_tickformat='.0%',
    height=560,
)
fig.write_html(STORY_DIR / 'fig_real_precinct_concentration_curve.html')
fig


### Figure 4. Real-precinct ticket share map

The map shows where the officer-written tickets are concentrated. It only shows the 78 mappable real precincts. Precinct 0 and a few unmappable precinct numbers are dropped.

We picked a choropleth because precincts are areas, not points. We used the YlOrRd color scale because the reader needs to see "more is darker". The midpoint is not meaningful, so a sequential scale is right.

The hover tooltip is rich. It shows the ticket count, the share of all real-precinct tickets, the dominant rule family, the average estimated fine, and the top 3 families inside that precinct. So the map is a small drill-down. A reader curious about one precinct can read its profile in one tooltip.

We use the carto-positron base map because it is gray and does not fight the choropleth color.


In [ ]:
geojson_path = OUTPUT_DIR / 'police_precincts.geojson'
with open(geojson_path) as f:
    precinct_geojson = json.load(f)
for feature in precinct_geojson['features']:
    feature['id'] = str(feature['properties']['precinct'])

real_precinct_map_profile = pd.read_csv(EDA_DIR / 'real_precinct_map_profile.csv', dtype={'precinct': str})
real_precinct_map_summary = load_json(STORY_DIR / 'real_precinct_map_summary.json')
real_precinct_map_profile.head(10)


In [ ]:
fig = px.choropleth_map(
    real_precinct_map_profile,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='id',
    color='share_of_real_precinct_tickets',
    color_continuous_scale='YlOrRd',
    hover_name='precinct',
    hover_data={
        'precinct': False,
        'tickets': ':,',
        'share_of_real_precinct_tickets': ':.2%',
        'dominant_violation_family': True,
        'average_estimated_fine': ':$.2f',
        'top_3_families': True,
    },
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.72,
    title='Real-precinct share of NYC curbside parking tickets',
    labels={
        'share_of_real_precinct_tickets': 'Share of real-precinct tickets',
        'tickets': 'Ticket count',
        'dominant_violation_family': 'Dominant family',
        'average_estimated_fine': 'Avg. estimated fine',
        'top_3_families': 'Top 3 families',
    },
)
fig.update_layout(
    margin=dict(l=0, r=0, t=55, b=0),
    coloraxis_colorbar=dict(tickformat='.1%', title='Ticket share'),
)
fig.write_html(STORY_DIR / 'fig_real_precinct_ticket_share_map.html')
fig


### Figure 5. Tickets per vehicle by plate class

This is the chart that supports the "It's not just where, it's who" section on the website. It moves the lens from precincts to vehicles.

Each bar is one plate class (passenger, commercial, taxi, and so on). The x axis is the average tickets per unique vehicle key in that class. The color is the share of vehicles in that class that got more than one ticket.

We picked a horizontal bar because the plate-class labels are long. The color encodes a second variable so the reader can see two ideas at once: how many tickets the average vehicle got, and how concentrated the ticketing is on repeat offenders.

We dropped plate classes with fewer than 1,000 unique vehicles. That removes noisy classes where one or two outlier vehicles distort the average.

Commercial and carrier plates sit at the top. Passenger plates are much lower. So the very high counts are working vehicles, not normal drivers.


In [ ]:
repeat_plate = pd.read_csv(EDA_DIR / 'repeat_vehicle_by_plate_class.csv')
repeat_plate_plot = repeat_plate[repeat_plate['vehicles'].ge(1000)].sort_values('tickets_per_vehicle')

fig = px.bar(
    repeat_plate_plot,
    x='tickets_per_vehicle', y='plate_class', orientation='h',
    color='repeat_vehicle_share', color_continuous_scale='Viridis',
    title='Tickets per vehicle by plate class',
)
fig.update_layout(
    xaxis_title='Tickets per unique vehicle key',
    yaxis_title='',
    coloraxis_colorbar={'title': 'Repeat share', 'tickformat': '.0%'},
    height=520,
)
fig.write_html(STORY_DIR / 'fig27_tickets_per_vehicle_plate_class.html')
fig


### Figure 6. Dominant family by precinct

This second map uses the same boundaries as Figure 4 but colors each precinct by its top family. So instead of a continuous scale, the color is a category.

We picked categorical colors because the variable is categorical. The reader can see a spatial pattern that is hidden in Figure 4. For example a sticker band in Staten Island. A meter band in Manhattan. The map answers "what kind of curb is this part of the city" instead of "how busy is it".

The hover keeps the same rich tooltip so the reader can also see the count.

Figure 4 and Figure 6 work as a pair. Figure 4 is "where". Figure 6 is "what kind". We keep the layout and base map the same so the eye is not distracted by style differences.


In [ ]:
fig = px.choropleth_map(
    real_precinct_map_profile,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='id',
    color='dominant_violation_family',
    hover_name='precinct',
    hover_data={
        'precinct': False,
        'tickets': ':,',
        'share_of_real_precinct_tickets': ':.2%',
        'dominant_family_share': ':.1%',
        'top_3_families': True,
        'average_estimated_fine': ':$.2f',
    },
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.75,
    title='Dominant parking-rule family by real precinct',
    labels={
        'dominant_violation_family': 'Dominant family',
        'tickets': 'Ticket count',
        'share_of_real_precinct_tickets': 'Share of real-precinct tickets',
        'dominant_family_share': 'Dominant family share',
        'top_3_families': 'Top 3 families',
        'average_estimated_fine': 'Avg. estimated fine',
    },
)
fig.update_layout(margin=dict(l=0, r=0, t=55, b=0), legend=dict(orientation='h', y=-0.05))
fig.write_html(STORY_DIR / 'fig_dominant_family_by_precinct_map.html')
fig


### Figure 7. Violation families by borough

This figure pulls the precinct-level pattern up to the borough level. Each group is one of the five NYC boroughs. The bars stack the top 8 violation families.

We picked a stacked bar because the natural question is "what mix of rules drives each borough". A stacked bar lets the reader compare the proportional shape across boroughs in one glance.

We chose borough as the grouping unit, not precinct, because the website needs a chart that anyone can read in two seconds. The full per-precinct view is in Figure 6.

Manhattan leans on meters and no-standing. The Bronx and Brooklyn lean on street cleaning. Staten Island stands out for sticker enforcement because the borough is car-dependent.


In [ ]:
family_borough = pd.read_csv(EDA_DIR / 'family_borough.csv')
top_families = family_totals.head(8)['family'].tolist()
family_borough_plot = family_borough[
    family_borough['borough'].isin(BOROUGHS) & family_borough['family'].isin(top_families)
]

fig = px.bar(
    family_borough_plot,
    x='borough', y='tickets', color='family', barmode='stack',
    title='Top violation families by borough',
)
fig.update_layout(xaxis_title='', yaxis_title='Tickets', legend_title='', height=560)
fig.write_html(STORY_DIR / 'fig10_family_by_borough.html')
fig


### Figure 8. Count vs estimated fine value

This figure is the dollar layer. Each point is one precinct. The x axis is ticket count. The y axis is estimated fine value. Color is the dominant family. Size is the average fine per ticket.

We picked a scatter plot because the question is "are these two numbers tracking each other". Most points line up. Some are above or below. The off-line points are the interesting ones.

We labeled the precincts that move the most when we switch from count rank to dollar rank. Precinct 33 is up 9 ranks. Precinct 78 is down 10. We did not label every point because that would be noisy.

Size encodes average fine per ticket. So a bigger circle means each ticket is worth more. The reader can see at a glance that double-parking-dominant precincts have bigger circles.


In [ ]:
real_precinct_fine_value = pd.read_csv(EDA_DIR / 'real_precinct_estimated_fine_value.csv', dtype={'precinct': str})
real_precinct_fine_summary = load_json(STORY_DIR / 'real_precinct_estimated_fine_value_summary.json')
real_precinct_fine_value.head(8)


In [ ]:
label_precincts = set(real_precinct_fine_value.head(8)['precinct'])
eligible_shift = real_precinct_fine_value[real_precinct_fine_value['ticket_count'] >= 25_000]
label_precincts.update(eligible_shift.sort_values('rank_shift', ascending=False).head(4)['precinct'])
label_precincts.update(eligible_shift.sort_values('rank_shift', ascending=True).head(4)['precinct'])
plot_df = real_precinct_fine_value.copy()
plot_df['label'] = plot_df['precinct'].where(plot_df['precinct'].isin(label_precincts), '')

fig = px.scatter(
    plot_df,
    x='ticket_count',
    y='estimated_total_fine_value',
    color='dominant_violation_family',
    size='average_estimated_fine_per_ticket',
    text='label',
    hover_name='precinct',
    hover_data={
        'precinct': False,
        'ticket_count': ':,',
        'estimated_total_fine_value': ':$,.0f',
        'average_estimated_fine_per_ticket': ':$.2f',
        'rank_by_ticket_count': True,
        'rank_by_estimated_fine_value': True,
        'rank_shift': True,
        'dominant_violation_family': True,
        'label': False,
    },
    title='Ticket counts and estimated fine value tell related but not identical stories',
    labels={
        'ticket_count': 'Ticket count',
        'estimated_total_fine_value': 'Estimated total fine value',
        'average_estimated_fine_per_ticket': 'Average estimated fine',
        'dominant_violation_family': 'Dominant violation family',
    },
)
fig.update_traces(textposition='top center')
fig.update_layout(
    yaxis_tickprefix='$',
    yaxis_tickformat=',.0f',
    xaxis_tickformat=',',
    height=650,
    legend=dict(orientation='h', y=-0.25),
)
fig.write_html(STORY_DIR / 'fig_real_precinct_count_vs_estimated_fine_value.html')
fig


### Figure 9. Hourly profiles by violation family

This is the time-of-day chart. Each line is one violation family. The y axis is the share of that family's daily tickets at each hour. So we are comparing the timing pattern of each family on the same scale.

We did not use raw counts because the families are very different in size. School zone speed has 4.8 million tickets. Hydrant has 660 thousand. Raw counts would let school zone dominate the chart. By normalizing within each family, all the lines are comparable.

The legend is interactive. The reader clicks a family name to hide or show it. This is bounded interactivity. The reader cannot change the data. They can only choose what to look at.

We pre-selected a default visible subset so the chart does not look like spaghetti on first load.


In [ ]:
violation_family_hourly = pd.read_csv(EDA_DIR / 'violation_family_hourly_profiles.csv')
hourly_profile_summary = load_json(STORY_DIR / 'violation_family_hourly_profiles_summary.json')
pd.read_csv(EDA_DIR / 'violation_family_hourly_peak_summary.csv')


In [ ]:
import numpy as np
import plotly.graph_objects as go

default_visible = hourly_profile_summary['default_visible_families']
families = hourly_profile_summary['families_in_figure']

fig = go.Figure()
for family in families:
    d = violation_family_hourly[violation_family_hourly['family'] == family].sort_values('hour')
    fig.add_trace(go.Scatter(
        x=d['hour'],
        y=d['share_of_family_daily_tickets'],
        mode='lines+markers',
        name=family,
        visible=True if family in default_visible else 'legendonly',
        customdata=np.stack([d['tickets'], d['share_of_family_daily_tickets']], axis=-1),
        hovertemplate=(
            'Family: ' + family + '<br>'
            'Hour: %{x}:00<br>'
            'Ticket count: %{customdata[0]:,}<br>'
            'Share of family daily tickets: %{customdata[1]:.1%}<extra></extra>'
        ),
    ))
fig.update_layout(
    title='Different parking rules have different daily rhythms',
    xaxis=dict(title='Hour of day', tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(title="Share of each family's valid-time tickets", tickformat='.0%'),
    height=650,
    legend=dict(orientation='h', y=-0.25),
)
fig.write_html(STORY_DIR / 'fig_violation_family_hourly_profiles.html')
fig


## 6. Discussion

This section is about what went well, what is missing, and what we could not answer.

### What went well

**The Precinct 0 finding.** This is the strongest part of the project. The first map was wrong because Precinct 0 was treated like a real place. Once we found the issue and separated cameras from officers, the rest of the analysis got much sharper. The website uses this as the first reveal because it has the biggest "aha" effect.

**The two-system framing.** Our story did not stop at "Precinct 0 is weird". We turned it into a positive claim. The dataset has two enforcement systems. Cameras and officers. Different daily clocks. Different rule families. Different boroughs lean on them differently. This made the story bigger than just a data cleaning note.

**The repeat-vehicle finding.** This was almost lost. It was a side step in the original notebook. We promoted it to its own section on the website. The numbers are striking. 56.6 percent of cars carry 89.2 percent of the tickets. One car got 1,088 tickets in a year. This connects the abstract "uneven curb" idea to a concrete fact.

**Honest claim discipline.** We never said "fair" or "unfair". We never said "revenue". We always said "issued tickets" and "estimated fine value". The captions repeat this on purpose. We think this is what makes the project feel honest.

### What is missing

**No exposure denominator.** The biggest missing piece is the bottom of the fraction. We can show how many tickets each precinct got. We cannot show how many vehicles parked there, how much curb each precinct has, or how many violations went unticketed. So we show the ticket footprint. We do not show the rate.

**No payment outcomes.** The dataset is the issued ticket file, not the paid ticket file. So our "estimated fine value" is just issued tickets times the rule book. The real number the city collects is smaller. Past city audits flagged more than a billion dollars in unpaid parking and camera fines. We mention this on the website but we cannot quantify it from this dataset.

**No demographics.** We cannot say anything about race, income, or neighborhood demographics. The dataset has plate, state, and vehicle type. It does not have driver identity. So claims about social fairness are not possible from this file alone.

**No street-level geography.** Our map stops at precinct level. A finer view would be by census tract, ZIP, or street segment. The dataset has street name and house number, but the geocoding is messy. We started a street analysis but did not finish it because the noise was too high.

**No causal claim.** We can show association, for example between double parking and narrow blocks. We cannot say one causes the other.

### Limitations

**The fine table is approximate.** We used the public NYC Department of Finance fine list. The exact dollar amount can change based on zone, repeat offender status, and other factors not in our file. So our 612 million dollar number is a reasonable upper-bound estimate, not a final figure.

**The borough cleaning is rough.** A small number of rows have unusual borough strings. We dropped them from borough charts but kept them in the totals.

**Precinct 0 is set aside, not deeply explored.** A separate project could be just about Precinct 0 (cameras only).

**One year only.** The website is FY2025. We did not look at year-over-year change. Adding more years would more than double the cleaning work.

### What would be next

If we had more time, we would add:

1. A street-level map for at least one borough. Broadway alone has 179,000 tickets. A map of the top 50 streets would let readers see where on each street the tickets land.
2. A year-over-year comparison. FY2023 vs FY2024 vs FY2025. If the patterns are stable, our story is strong. If they are changing, that itself is the story.
3. A small social-data sharing layer. The Segel and Heer paper says this is under-used in narrative visualizations. Readers could add comments next to a precinct. This would make the project feel more like a public conversation.


## 7. Contributions

This section says who did which parts of the project.

### S.T. Hassan (s250112)

- Owned the data cleaning pipeline. Wrote the chunked reader for the 3 GB CSV.
- Built the Precinct 0 separation logic.
- Made the violation family grouping.
- Owned the website. Wrote the HTML, CSS, and JavaScript.
- Owned the time-of-day analysis. Built the hourly profile chart.
- Did the concentration curve and the precinct specialization analysis.
- Did the repeat vehicle analysis.

### M. Abbas Khan (s250145)

- Owned the geographic analysis. Joined the precinct GeoJSON.
- Built the choropleth maps.
- Made the dominant family map and the ticket share map.
- Wrote the public-facing prose.
- Built the embedded SVG diagram and the neighborhood callout cards.

### Shared work

- The story structure was decided in group meetings. All members agreed on the section order and the claim discipline.
- The reference list was assembled jointly. Each member added the sources they used.
- This notebook was reviewed by both members before submission.


## 8. References

This section lists the sources we used. We split them into three groups: primary data, official rule sources, and framing sources.

### Primary data

1. **Parking Violations Issued, Fiscal Year 2025.** NYC Open Data. The full ticket records used in this project. Link: https://data.cityofnewyork.us/d/m5vz-tzqv

2. **Police Precincts.** NYC Open Data, NYC Department of City Planning. Precinct boundary GeoJSON used for all maps. Link: https://data.cityofnewyork.us/d/y76i-bdw7

3. **Borough Boundaries.** NYC Open Data. GeoJSON for the borough map. Link: https://data.cityofnewyork.us/City-Government/Borough-Boundaries/tqmj-j8zm

### Official rule sources

4. **Annual Report of NYC Parking Tickets and Camera Violations: Fiscal Year 2025.** NYC Department of Finance, 2025. Official city totals. Link: https://www.nyc.gov/assets/finance/downloads/pdf/25pdf/2025-local-law-6-report.pdf

5. **Violation Codes, Fines, Rules, and Regulations.** NYC Department of Finance. Code meanings and fine table used in fine estimation. Link: https://www.nyc.gov/site/finance/vehicles/services-violation-codes.page

6. **Street Cleaning and Alternate Side Parking.** NYC Department of Sanitation. Background on the alternate side schedule. Link: https://www.nyc.gov/site/dsny/what-we-do/cleaning/street-cleaning-asp.page

7. **Parking Meters.** NYC Department of Transportation. Background on metered parking. Link: https://www.nyc.gov/html/dot/html/motorist/parking-rates.shtml

8. **Automated Camera Enforcement.** Metropolitan Transportation Authority. Background on MTA bus-mounted cameras. Link: https://www.mta.info/agency/new-york-city-transit/automated-camera-enforcement

9. **Speed Cameras Frequently Asked Questions.** NYC Department of Transportation. Background on school-zone speed cameras. Link: https://www.nyc.gov/html/dot/downloads/pdf/speed-camera-faq.pdf

10. **Red Light Cameras.** NYC311. Background on red-light camera records. Link: https://portal.311.nyc.gov/article/?kanumber=KA-02326

### Framing and methodology

11. **Segel, E. and Heer, J. (2010).** Narrative Visualization: Telling Stories with Data. *IEEE Transactions on Visualization and Computer Graphics, 16*(6). The framework we used to choose the genre and to map our project onto Figure 7.

12. **What Parking Ticket Data Can and Cannot Tell Us.** Urban Institute, 2020. Framing for the limits of the data. Link: https://www.urban.org/urban-wire/what-parking-ticket-data-can-and-cannot-tell-us-amid-calls-reform-fines-and-fees

13. **Curb Management.** NACTO. Background on the curb as a managed city resource. Link: https://nacto.org/program/reimagining-city-streets/multimodal-streets/curb-management/

14. **Look for Inequities in Parking Tickets.** Dan Levine. Careful wording for geographic ticket analysis. Link: https://danlevine.work/projects/look-for-inequities-in-parking-tickets/

15. **NYC Parking Tickets: City Is Owed More Than 2 Billion in Unpaid Fines.** NBC New York, 2023. Source for the "estimated, not collected" framing.

### Neighborhood context

16. **NYPD precinct pages.** Each NYPD precinct has an official page on nyc.gov/site/nypd. We used these to identify what neighborhood each precinct covers. For example Precinct 19 (Upper East Side), Precinct 14 (Midtown South), Precinct 34 (Washington Heights), Precinct 44 (South Bronx), Precinct 123 (South Shore Staten Island).

17. **Wikipedia: Upper East Side.** Background on the dominant precinct in our data. Link: https://en.wikipedia.org/wiki/Upper_East_Side

### Project files

18. **The Uneven Curb (project website).** DTU Social Visualization, 2026. Link: https://syedtaimurhassan.github.io/02806-project/site/

19. **Project repository on GitHub.** Link: https://github.com/syedtaimurhassan/02806-project

20. **Parking Violations Issued Data Dictionary.** NYC OpenData field reference, included in the repo as `Parking_Violations_Issued_Data_Dictionary.xlsx`.
